<a href="https://colab.research.google.com/github/NOUSHEENSHAIK11/Elite-tech-task/blob/main/Task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import math
import time
import random
from typing import Optional



def gpt2_generate(prompt: str,
                  max_new_tokens: int = 200,
                  temperature: float = 0.8,
                  top_k: int = 50,
                  top_p: float = 0.95,
                  num_return_sequences: int = 1,
                  model_name: str = "gpt2") -> list[str]:

    try:
        from transformers import pipeline, set_seed
    except ImportError:
        return ["❌  Please run: pip install transformers torch"]

    print(f"\n⏳  Loading {model_name} (first run downloads the model) …")
    generator = pipeline(
        "text-generation",
        model=model_name,
        device=-1,          # use CPU; change to 0 for GPU
    )
    set_seed(42)

    outputs = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        pad_token_id=50256,   # EOS token for GPT-2
    )
    return [o["generated_text"] for o in outputs]




try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False


class CharLSTM(nn.Module if TORCH_AVAILABLE else object):


    def __init__(self, vocab_size: int, embed_dim: int = 64,
                 hidden_dim: int = 256, num_layers: int = 2,
                 dropout: float = 0.3):
        if not TORCH_AVAILABLE:
            raise RuntimeError("PyTorch not installed.")
        super().__init__()
        self.embed      = nn.Embedding(vocab_size, embed_dim)
        self.lstm       = nn.LSTM(embed_dim, hidden_dim, num_layers,
                                  batch_first=True, dropout=dropout)
        self.fc         = nn.Linear(hidden_dim, vocab_size)
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

    def forward(self, x, hidden=None):
        x = self.embed(x)
        out, hidden = self.lstm(x, hidden)
        logits = self.fc(out)
        return logits, hidden

    def init_hidden(self, batch_size: int, device):
        return (torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device),
                torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device))


class CharLSTMTrainer:


    SEQ_LEN    = 100
    BATCH_SIZE = 64
    EPOCHS     = 20
    LR         = 0.002
    EMBED_DIM  = 64
    HIDDEN_DIM = 256
    NUM_LAYERS = 2

    def __init__(self):
        if not TORCH_AVAILABLE:
            raise RuntimeError("PyTorch not installed.")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model  = None
        self.char2idx: dict[str, int] = {}
        self.idx2char: dict[int, str] = {}

    def _build_vocab(self, text: str):
        chars = sorted(set(text))
        self.char2idx = {c: i for i, c in enumerate(chars)}
        self.idx2char = {i: c for c, i in self.char2idx.items()}

    def _encode(self, text: str):
        import torch
        return torch.tensor([self.char2idx[c] for c in text],
                             dtype=torch.long)

    def train(self, corpus: str, epochs: int = EPOCHS,
              verbose: bool = True) -> "CharLSTM":
        import torch
        self._build_vocab(corpus)
        vocab_size = len(self.char2idx)
        data       = self._encode(corpus).to(self.device)

        self.model = CharLSTM(vocab_size, self.EMBED_DIM,
                               self.HIDDEN_DIM, self.NUM_LAYERS).to(self.device)
        optimizer  = torch.optim.Adam(self.model.parameters(), lr=self.LR)
        criterion  = nn.CrossEntropyLoss()

        n_seqs = (len(data) - 1) // self.SEQ_LEN
        X = torch.stack([data[i * self.SEQ_LEN: i * self.SEQ_LEN + self.SEQ_LEN]
                         for i in range(n_seqs)])
        Y = torch.stack([data[i * self.SEQ_LEN + 1: i * self.SEQ_LEN + self.SEQ_LEN + 1]
                         for i in range(n_seqs)])

        dataset   = torch.utils.data.TensorDataset(X, Y)
        loader    = torch.utils.data.DataLoader(dataset,
                                                batch_size=self.BATCH_SIZE,
                                                shuffle=True)

        print(f"\n🚂  Training CharLSTM on {self.device} "
              f"({vocab_size} chars, {epochs} epochs) …")
        for epoch in range(1, epochs + 1):
            total_loss = 0.0
            self.model.train()
            for xb, yb in loader:
                xb, yb  = xb.to(self.device), yb.to(self.device)
                hidden  = self.model.init_hidden(xb.size(0), self.device)
                optimizer.zero_grad()
                logits, _ = self.model(xb, hidden)
                loss = criterion(logits.reshape(-1, vocab_size),
                                 yb.reshape(-1))
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 5.0)
                optimizer.step()
                total_loss += loss.item()

            if verbose and epoch % 5 == 0:
                perplexity = math.exp(total_loss / len(loader))
                print(f"  Epoch {epoch:>3}/{epochs}  "
                      f"loss={total_loss / len(loader):.4f}  "
                      f"perplexity={perplexity:.1f}")

        print("✅  Training complete.")
        return self.model

    def generate(self, seed: str = " ", length: int = 400,
                 temperature: float = 0.8) -> str:

        if self.model is None:
            return "❌  Model not trained yet – call .train(corpus) first."
        import torch

        self.model.eval()
        # Replace unknown chars in seed
        seed = "".join(c if c in self.char2idx else " " for c in seed)
        if not seed.strip():
            seed = " "

        input_ids = self._encode(seed).unsqueeze(0).to(self.device)
        hidden    = self.model.init_hidden(1, self.device)
        generated = seed

        with torch.no_grad():
            # Prime the hidden state
            _, hidden = self.model(input_ids, hidden)
            # Generate new characters
            last_char = input_ids[:, -1:]
            for _ in range(length):
                logits, hidden = self.model(last_char, hidden)
                logits = logits[:, -1, :] / temperature
                probs  = torch.softmax(logits, dim=-1)
                next_idx = torch.multinomial(probs, num_samples=1)
                char     = self.idx2char[next_idx.item()]
                generated += char
                last_char  = next_idx

        return generated



SAMPLE_CORPUS = """
Artificial intelligence is transforming the world in remarkable ways. Machine learning
algorithms can now recognise images, understand speech, and generate text with human-like
fluency. Deep neural networks, inspired by the structure of the human brain, have achieved
superhuman performance on tasks once thought to require genuine intelligence.

Natural language processing allows computers to understand and generate human language.
Models like GPT and BERT have revolutionised how machines interact with text. These models
are trained on billions of words from the internet and learn statistical patterns that
capture grammar, facts, and even reasoning ability.

The future of AI holds tremendous promise. Self-driving cars, personalised medicine,
and climate modelling are just a few areas where AI is already making a difference.
As algorithms become more powerful and datasets grow larger, the possibilities expand
further. Researchers continue to explore new architectures and training techniques to
push the boundaries of what machines can learn and do.

Ethics and safety are critical considerations in the development of artificial intelligence.
Ensuring that AI systems are fair, transparent, and aligned with human values requires
careful design and robust evaluation. Collaboration between technologists, policymakers,
and the public is essential to ensure that the benefits of AI are shared widely and its
risks are managed responsibly.
""" * 10      # repeat to give the LSTM enough data



TOPIC_PROMPTS = {
    "1": ("Artificial Intelligence",
          "Artificial intelligence is transforming"),
    "2": ("Climate Change",
          "Climate change poses one of the greatest challenges"),
    "3": ("Space Exploration",
          "The exploration of space has always"),
    "4": ("Renewable Energy",
          "Renewable energy sources such as solar and wind"),
    "5": ("Custom",  None),
}


def run_demo():
    print("=" * 65)

    print("=" * 65)

    print("\nChoose a generation backend:")
    print("  [1] GPT-2 via HuggingFace Transformers  (recommended)")
    print("  [2] Character-level LSTM  (trains from scratch, no internet needed)")
    choice = input("\nEnter 1 or 2 [default 1]: ").strip() or "1"

    # ── GPT-2 path ────────────────────────────────────────────────
    if choice == "1":
        print("\nSelect a topic prompt:")
        for k, (name, _) in TOPIC_PROMPTS.items():
            print(f"  [{k}] {name}")
        topic_choice = input("\nEnter topic number [default 1]: ").strip() or "1"

        name, prompt = TOPIC_PROMPTS.get(topic_choice, TOPIC_PROMPTS["1"])
        if prompt is None:
            prompt = input("Enter your own prompt: ").strip()
            name   = "Custom"

        try:
            length = int(input("Max new tokens to generate [default 200]: ") or 200)
            temp   = float(input("Temperature 0.1–1.5 [default 0.8]: ") or 0.8)
            n_seq  = int(input("Number of outputs [default 1]: ") or 1)
        except ValueError:
            length, temp, n_seq = 200, 0.8, 1

        print(f"\n📝  Topic  : {name}")
        print(f"🌱  Prompt : {prompt!r}\n")

        results = gpt2_generate(
            prompt,
            max_new_tokens=length,
            temperature=temp,
            num_return_sequences=n_seq,
        )
        for i, text in enumerate(results, 1):
            print(f"\n{'─'*65}")
            print(f"✅  Generated text {i}/{len(results)}:\n")
            print(text)


    elif choice == "2":
        if not TORCH_AVAILABLE:
            print("❌  PyTorch not found. Run: pip install torch")
            return

        print("\nThe LSTM will train on a built-in AI-themed corpus.")
        try:
            epochs = int(input("Training epochs [default 20]: ") or 20)
            length = int(input("Characters to generate [default 400]: ") or 400)
            temp   = float(input("Temperature 0.1–1.5 [default 0.8]: ") or 0.8)
        except ValueError:
            epochs, length, temp = 20, 400, 0.8

        seed = input("Seed text [default='Artificial intelligence']: ").strip() \
               or "Artificial intelligence"

        trainer = CharLSTMTrainer()
        trainer.train(SAMPLE_CORPUS, epochs=epochs)

        print(f"\n📝  Seed: {seed!r}\n{'─'*65}")
        generated = trainer.generate(seed, length=length, temperature=temp)
        print(f"✅  Generated text:\n\n{generated}")

    else:
        print("❌  Invalid choice.")
        return

    print("\n" + "=" * 65)

    print("=" * 65)


if __name__ == "__main__":
    run_demo()


Choose a generation backend:
  [1] GPT-2 via HuggingFace Transformers  (recommended)
  [2] Character-level LSTM  (trains from scratch, no internet needed)

Enter 1 or 2 [default 1]: 1

Select a topic prompt:
  [1] Artificial Intelligence
  [2] Climate Change
  [3] Space Exploration
  [4] Renewable Energy
  [5] Custom

Enter topic number [default 1]: 1
Max new tokens to generate [default 200]: 150
Temperature 0.1–1.5 [default 0.8]: 0.8
Number of outputs [default 1]: 1

📝  Topic  : Artificial Intelligence
🌱  Prompt : 'Artificial intelligence is transforming'


⏳  Loading gpt2 (first run downloads the model) …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'do_sample', 'top_k', 'max_new_tokens', 'temperature', 'num_return_sequences', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



─────────────────────────────────────────────────────────────────
✅  Generated text 1/1:

Artificial intelligence is transforming our lives," says Dr. Charles S. Dominguez, Director of the Center for Information and Communication Technology at the University of North Carolina at Chapel Hill and a senior scientist at Intel Corporation. "As people become more sophisticated, we'll be able to apply this knowledge to ways to reduce our reliance on our phones and computers."

As more and more people realize their phone and laptop are far more powerful than they ever thought, this technology could help to shift the global landscape in which we interact with our world.

The breakthrough is that researchers in Japan, China and South Korea are developing "smart-phone devices that can detect and control the motion of our mobile devices," Dr. Dominguez says. They will also be able to

